In [0]:
import logging
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO)

try:
    logging.info("Dashboard summary started")

    df = spark.table("fraud_catalog.gold.fraud_transactions")

    logging.info(f"Gold rows read: {df.count()}")

except Exception as e:
    logging.error(str(e))
    raise

INFO:root:Dashboard summary started
INFO:root:Gold rows read: 1048575


In [0]:
fraud_by_date = (
    df.groupBy("txn_date").agg(F.count("*").alias("total_txn"),F.sum("fraud_flag").alias("fraud_count"))
      .withColumn("fraud_pct", F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2)))


fraud_by_date.write \
  .format("delta") \
  .mode("overwrite") \
  .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/fraud_by_date") \
  .saveAsTable("fraud_catalog.gold.fraud_by_date")

logging.info("fraud_by_date table created")

INFO:root:fraud_by_date table created


In [0]:
display(spark.table("fraud_catalog.gold.fraud_by_date"))

txn_date,total_txn,fraud_count,fraud_pct
2012-07-14,3878,27,0.7
2012-09-01,4129,28,0.68
2012-08-05,3907,23,0.59
2012-08-11,4089,25,0.61
2012-12-22,6325,71,1.12
2012-08-18,3940,38,0.96
2012-07-22,3830,25,0.65
2012-12-15,6425,90,1.4
2012-12-26,3639,23,0.63
2012-07-28,4045,40,0.99


In [0]:
fraud_by_city = (df.groupBy("city").agg(F.count("*").alias("total_txn"),F.sum("fraud_flag").alias("fraud_count"))
      .withColumn("fraud_pct", F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2)))

(fraud_by_city.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/fraud_by_city") \
    .saveAsTable("fraud_catalog.gold.fraud_by_city"))

logging.info("fraud_by_city table created")

INFO:root:fraud_by_city table created


In [0]:
display(spark.table("fraud_catalog.gold.fraud_by_city").orderBy(F.desc("fraud_pct")))

city,total_txn,fraud_count,fraud_pct
North East,9,5,55.56
Wappapello,8,4,50.0
Mount Vernon,11,5,45.45
La Grande,12,5,41.67
Irvington,8,3,37.5
Coulee Dam,15,5,33.33
Byesville,12,4,33.33
Hubbell,19,6,31.58
Nanuet,10,3,30.0
Roland,11,3,27.27


In [0]:
fraud_by_category = (df.groupBy("category").agg(F.count("*").alias("total_txn"),F.sum("fraud_flag").alias("fraud_count"))
      .withColumn("fraud_pct", F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2)))

(fraud_by_category.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/fraud_by_category") \
    .saveAsTable("fraud_catalog.gold.fraud_by_category"))

logging.info("fraud_by_category table created")

INFO:root:fraud_by_category table created


In [0]:
display(fraud_by_category.orderBy(F.col("fraud_count").desc()))

category,total_txn,fraud_count,fraud_pct
shopping_net,78899,1319,1.67
shopping_pos,94353,1009,1.07
gas_transport,106430,378,0.36
misc_pos,64492,370,0.57
grocery_pos,99906,367,0.37
misc_net,51082,353,0.69
travel,32830,245,0.75
entertainment,75981,215,0.28
home,99578,201,0.2
kids_pets,91404,165,0.18


In [0]:
fraud_by_hour = (df.groupBy("txn_hour").agg(F.count("*").alias("total_txn"),F.sum("fraud_flag").alias("fraud_count"))
      .withColumn("fraud_pct", F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2)))

(fraud_by_hour.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/fraud_by_hour") \
    .saveAsTable("fraud_catalog.gold.fraud_by_hour"))

logging.info("fraud_by_hour table created")

INFO:root:fraud_by_hour table created


In [0]:
display(fraud_by_hour.orderBy("txn_hour"))

txn_hour,total_txn,fraud_count,fraud_pct
0,34412,300,0.87
1,34757,314,0.9
2,34407,297,0.86
3,34556,341,0.99
4,33943,291,0.86
5,34106,287,0.84
6,34316,22,0.06
7,34158,16,0.05
8,34256,18,0.05
9,34006,12,0.04


In [0]:
top_risk_customers = (df.groupBy("cc_num").agg(F.count("*").alias("total_txn"),
          F.sum("fraud_flag").alias("fraud_count"),
          F.round(F.sum("amt"), 2).alias("total_amount"))
      .withColumn("fraud_pct",F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2)))

(top_risk_customers.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/top_risk_customers")
    .saveAsTable("fraud_catalog.gold.top_risk_customers"))

logging.info("top_risk_customers table created")

INFO:root:top_risk_customers table created


In [0]:
display(top_risk_customers.orderBy(F.col("fraud_count").desc()))

cc_num,total_txn,fraud_count,total_amount,fraud_pct
6.01137E+15,4173,62,328717.19,1.49
4.30248E+15,3317,35,213741.42,1.06
6.53463E+15,2542,34,135884.1,1.34
1.80065E+14,2500,33,163452.81,1.32
3.51867E+15,2034,31,109045.03,1.52
2.13174E+14,2474,30,167747.29,1.21
2.72043E+15,2559,28,138531.56,1.09
1.80048E+14,2513,26,224944.45,1.03
3.54511E+15,2531,25,121377.87,0.99
3.58364E+15,2493,25,230548.67,1.0


In [0]:
top_risk_merchants = (df.groupBy("merchant").agg(
          F.count("*").alias("total_txn"),
          F.sum("fraud_flag").alias("fraud_count"),
          F.round(F.sum("amt"), 2).alias("total_amount"))
      .withColumn("fraud_pct",F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2)))

(top_risk_merchants.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/top_risk_merchants")
    .saveAsTable("fraud_catalog.gold.top_risk_merchants"))

logging.info("top_risk_merchants table created")

INFO:root:top_risk_merchants table created


In [0]:
display(top_risk_merchants.orderBy(F.col("fraud_count").desc()))

merchant,total_txn,fraud_count,total_amount,fraud_pct
fraud_Price Inc,1618,43,170998.37,2.66
fraud_Rempel Inc,1580,39,155947.62,2.47
"fraud_Kuhic, Bins and Pfeffer",1614,38,156662.47,2.35
fraud_Kuhic LLC,1593,37,149356.43,2.32
fraud_Schmidt and Sons,1579,35,136605.94,2.22
"fraud_Schmeler, Bashirian and Price",1583,32,153310.32,2.02
"fraud_Greenholt, O'Hara and Balistreri",1578,32,137042.73,2.03
fraud_Mohr-Bayer,1634,31,137924.22,1.9
"fraud_Towne, Greenholt and Koepp",1575,31,154139.1,1.97
fraud_Kozey-Boehm,1513,31,150114.12,2.05


In [0]:
fraud_kpi = df.agg(F.count("*").alias("total_txn"),F.sum("fraud_flag").alias("fraud_count")).withColumn("fraud_pct",F.round((F.col("fraud_count") / F.col("total_txn")) * 100, 2))

fraud_kpi.write \
  .format("delta") \
  .mode("overwrite") \
  .option("path", "abfss://fraudexternal@fraudstorageacc1234.dfs.core.windows.net/gold/fraud_kpi") \
  .saveAsTable("fraud_catalog.gold.fraud_kpi")

logging.info("fraud_kpi table created")

INFO:root:fraud_kpi table created


In [0]:
display(fraud_kpi)

total_txn,fraud_count,fraud_pct
1048575,5203,0.5
